# 01 Check TRF-Tools Pipeline Setup

This notebook checks the TRF-Tools pipeline inputs before estimating any TRFs. It is meant to be run cell-by-cell.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PIPELINE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from alice_trf_experiment import BIDS_ROOT, SEGMENT_DURATION, alice

print(f'Pipeline directory: {PIPELINE_DIR}')
print(f'BIDS root: {BIDS_ROOT}')

## Subjects and Segment Durations

In [ ]:
subjects = alice.get_field_values('subject')
print(f'N subjects: {len(subjects)}')
print(subjects[:10], '...', subjects[-5:])

duration_table = pd.DataFrame({
    'segment': list(SEGMENT_DURATION.keys()),
    'duration_sec': list(SEGMENT_DURATION.values()),
}).sort_values('segment', key=lambda s: s.astype(int))
duration_table

## Predictor Files

The first formal model uses `gammatone-8` files in BIDS derivatives.

In [ ]:
predictor_dir = BIDS_ROOT / 'derivatives' / 'predictors'
predictor_rows = []
for segment in sorted(SEGMENT_DURATION, key=int):
    path = predictor_dir / f'{segment}~gammatone-8.pickle'
    predictor_rows.append({'segment': segment, 'path': str(path), 'exists': path.exists()})

predictor_table = pd.DataFrame(predictor_rows)
display(predictor_table)
assert predictor_table['exists'].all(), 'Missing gammatone-8 predictor files'

## Events for One Subject

TRF-Tools/Eelbrain reads BrainVision raw markers. The pipeline maps raw marker strings to the clean `segment` variable.

In [ ]:
subject = '01'
events = alice.load_events(subject)
print(f'Loaded {events.n_cases} events for subject {subject}')
events.head()

In [ ]:
print('Raw event labels:', list(events['event']))
print('Clean segment labels:', list(events['segment']))

## Channel-Type Check

`AUD` should remain in raw data as a `misc` channel, not as an EEG target.

In [ ]:
alice.set(subject='01', raw='0.5-20')
raw = alice.load_raw(preload=False)
channel_types = raw.get_channel_types()
print(f'N channels total: {len(raw.ch_names)}')
print(f'N EEG channels: {channel_types.count("eeg")}')
print(f'N MISC channels: {channel_types.count("misc")}')
print(f'AUD present: {"AUD" in raw.ch_names}')
if 'AUD' in raw.ch_names:
    print(f'AUD type: {raw.get_channel_types(picks=["AUD"])[0]}')